# Clustering Comparison: make_moons

Comparing all 5 methods on `sklearn.datasets.make_moons()` (1,000 points,
2 interleaving half-moon classes, `noise=0.07`). This synthetic dataset is
specifically designed to be **non-convex**, which is expected to separate
methods that assume convex/Gaussian clusters (K-Means++, GMM) from those
that don't (DBSCAN, HDBSCAN, Spectral Clustering with a suitable affinity).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sc4020 import generate_figures as gf
from sc4020 import methods, tuning
from sc4020.datasets import make_moons
from sc4020.paths import results_dir

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
METRIC_COLS = [
    "method", "fit_seconds", "n_clusters", "noise_fraction", "largest_cluster_share",
    "silhouette", "davies_bouldin", "calinski_harabasz",
]
EXTERNAL_COLS = ["ari", "nmi", "ami", "fmi"]

In [ ]:
ds = make_moons(n_samples=1000, noise=0.07, random_state=42)
print(f"{ds.name}: X={ds.data.shape}, classes={ds.target_names}, n_true_clusters={ds.n_true_clusters}")

## Exploratory look

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(ds.data[:, 0], ds.data[:, 1], c=ds.target, cmap="tab10", s=15)
ax.set_title("make_moons ground truth")
plt.show()

## 2D embedding for visualization

The data is already 2D, so no dimensionality reduction is needed for plotting
— the standardized features are used directly, both for clustering and viz.

In [ ]:
X_2d = ds.data

## DBSCAN eps selection

In [ ]:
ms = tuning.default_min_samples(ds.data.shape[1])
curve = tuning.k_distance_curve(ds.data, ms)
knee_idx, eps_at_knee = tuning.knee(curve)
gf.plot_k_distance(curve, knee_idx, eps_at_knee, min_samples=ms)
plt.show()

## Fit all 5 methods

K = 2 (true class count). Spectral Clustering uses a nearest-neighbors
affinity — required to respect the moons' manifold shape (an RBF affinity
would blur the two interleaving arcs together).

In [ ]:
# The shared rule (tuning.select_all): K-Means k by silhouette (the number of classes when known),
# GMM by BIC, DBSCAN and HDBSCAN by the best silhouette with >= 2 clusters and <= 50% noise.
# Spectral clustering runs at the K-Means k.
selections = tuning.select_all(ds.data, k=ds.n_true_clusters, y_true=ds.target)
params = {m: s.params for m, s in selections.items()}
params["Spectral"] = {"n_clusters": params["K-Means"]["n_clusters"], "affinity": "nearest_neighbors"}
results = methods.run_models(methods.build_models(params), ds.data, params=params)
for name, r in results.items():
    print(f"{name:22s} params={r.params}  fit_seconds={r.fit_seconds:.3f}")

## Metrics

In [ ]:
results_df = gf.build_results_table(ds.name, ds.data, results, ds.target)
results_df.to_csv(results_dir("datasets") / "make_moons_results.csv", index=False)
display(results_df[METRIC_COLS])
display(results_df[["method"] + EXTERNAL_COLS])

## Cluster visualizations vs. ground truth

In [ ]:
fig = gf.plot_cluster_grid(ds.data, results, y_true=ds.target, class_names=ds.target_names,
                           title="make_moons: predicted clusters vs. ground truth")
plt.show()

## Discussion

- **Spectral Clustering (nearest-neighbors affinity), DBSCAN, and HDBSCAN**
  should all recover the two moons almost perfectly (ARI/NMI close to 1.0):
  they don't assume convex clusters, so they can follow the curved,
  interleaving shape.
- **K-Means++ and GMM** should score much lower: both assume roughly
  spherical/Gaussian clusters, so they instead split the data with a straight
  decision boundary that cuts across both moons — clearly visible as
  mismatched colors relative to the ground-truth panel.
- Note that internal metrics (Silhouette, Davies-Bouldin, Calinski-Harabasz)
  can actually favor K-Means++/GMM here even though their *external* metrics
  (ARI/NMI) are worse — internal metrics assume convex, well-separated
  clusters too, so they reward the "wrong but convex" answer. This is a good
  illustration of why external metrics matter whenever ground truth is
  available, and why no single metric tells the whole story.